<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 7: Musteri Terk Analizi

**MAKİNE ÖĞRENMESİ UZMANLIĞI** · Modül 7 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta07/hafta07_musteri_terk_analizi.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta07/hafta07_musteri_terk_analizi.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>

</div>

# Hafta 7 — Müşteri Terk (Churn) Analizi

Bu defterde gerçek IBM Telco verisi üzerinde **XGBoost** ile müşteri terk tahmini yapacağız.

## İçindekiler
1. Kütüphaneler
2. Gerçek Veri Seti Yükleme (IBM Telco Churn)
3. Keşifsel Veri Analizi (EDA)
4. Özellik Mühendisliği ve Kodlama
5. XGBoost Model Eğitimi
6. Özellik Önemi
7. Model Değerlendirme
8. İş Analitiği ve Öneriler

## Churn (Müşteri Terk) Nedir?

**Churn**, bir müşterinin hizmetten ayrılması (aboneliğini iptal etmesi) durumudur. Telekomünikasyon, bankacılık ve SaaS sektörlerinde en kritik iş metriklerinden biridir.

- Yeni müşteri kazanmak, mevcut müşteriyi tutmaktan **5-7 kat daha pahalıdır**
- Terk oranını %5 düşürmek, kârı **%25-95** artırabilir
- Bu yüzden şirketler ML modelleriyle **terk riski yüksek müşterileri önceden tespit edip** önlem almaya çalışır

## 1. Kütüphanelerin Yüklenmesi

| Kütüphane | Amacı |
|-----------|-------|
| `xgboost` | Gradient Boosting tabanlı güçlü sınıflandırma algoritması |
| `sklearn.preprocessing.LabelEncoder` | Kategorik değişkenleri sayıya çevirme |
| `sklearn.metrics` | Karışıklık matrisi, sınıflandırma raporu, ROC/AUC |

> **XGBoost** (eXtreme Gradient Boosting), Kaggle yarışmalarında en çok kullanılan algoritmadır. Zayıf öğrenicileri (karar ağaçları) sırayla birleştirerek güçlü bir model oluşturur.

In [ ]:
!pip install -q xgboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, auc
)
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("Kütüphaneler yüklendi!")

## 2. Gerçek IBM Telco Customer Churn Verisini Yükleme

**IBM Telco Customer Churn** — 7.043 müşterinin gerçek telekom abonelik verileri.

**Kaynak:** [Kaggle - Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

### Sütun Açıklamaları

| Sütun | Açıklama | Tip |
|-------|----------|-----|
| `customerID` | Benzersiz müşteri kimliği | ID |
| `gender` | Cinsiyet (Male/Female) | Kategorik |
| `SeniorCitizen` | 65 yaş üstü mü? (0/1) | Binary |
| `tenure` | Müşteri süresi (ay) | Sayısal |
| `Contract` | Sözleşme türü (Aylık/1 Yıl/2 Yıl) | Kategorik |
| `MonthlyCharges` | Aylık ücret ($) | Sayısal |
| `TotalCharges` | Toplam ödenen tutar ($) | Sayısal |
| `InternetService` | İnternet hizmet türü (DSL/Fiber/Yok) | Kategorik |
| `PaymentMethod` | Ödeme yöntemi | Kategorik |
| `TechSupport` | Teknik destek var mı? | Kategorik |
| **`Churn`** | **Müşteri terk etti mi? (Yes/No)** | **Hedef** |

> **Veri Temizleme Notu:** `TotalCharges` sütununda bazı boşluk değerleri var — bunları sayıya çevirip medyanla dolduruyoruz.

In [ ]:
# IBM Telco Customer Churn Dataset (gerçek veri)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# TotalCharges'da boşluk olan değerler var — sayıya çevir
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Churn sütununu 0/1'e çevir (model için)
df['Churn_Flag'] = (df['Churn'] == 'Yes').astype(int)

print(f"Veri seti boyutu: {df.shape}")
print(f"\nChurn dağılımı:")
print(df['Churn'].value_counts())
print(f"\nChurn oranı: {df['Churn_Flag'].mean():.2%}")
print(f"Eksik veri: {df.isnull().sum().sum()}")
df.head()

## 3. Keşifsel Veri Analizi (EDA)

### 3.1 Genel Terk Dağılımı ve Temel Faktörler

Aşağıdaki 4 grafikte terk davranışını etkileyen ana faktörlere bakıyoruz:

1. **Pasta grafiği:** Genel terk oranı (dengesiz sınıf dağılımı var mı?)
2. **Sözleşme türü:** Aylık vs yıllık sözleşmelerde terk farkı
3. **Müşteri süresi (tenure):** Yeni mi eski mi müşteriler terk ediyor?
4. **Aylık ücret:** Yüksek ücret ödeyenler daha mı çok terk ediyor?

> **Yeşil** = Kalan müşteriler, **Kırmızı** = Terk eden müşteriler

In [ ]:
# Terk dağılımı — 4 farklı perspektif
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Genel terk dağılımı (pasta grafiği)
churn_counts = df['Churn'].value_counts()
axes[0, 0].pie(churn_counts, labels=['Kalan', 'Terk Eden'], autopct='%1.1f%%',
               colors=['#2ecc71', '#e74c3c'], startangle=90, textprops={'fontsize': 13})
axes[0, 0].set_title('Müşteri Terk Dağılımı')

# 2. Sözleşme türüne göre terk oranı
ct = df.groupby('Contract')['Churn_Flag'].mean().sort_values(ascending=False)
ct.plot(kind='bar', ax=axes[0, 1], color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='white')
axes[0, 1].set_title('Sözleşme Türüne Göre Terk Oranı')
axes[0, 1].set_ylabel('Terk Oranı')
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)

# 3. Müşteri süresi dağılımı (terk durumuna göre)
df[df['Churn'] == 'No']['tenure'].hist(ax=axes[1, 0], bins=30, alpha=0.6, 
                                        color='#2ecc71', label='Kalan')
df[df['Churn'] == 'Yes']['tenure'].hist(ax=axes[1, 0], bins=30, alpha=0.6, 
                                         color='#e74c3c', label='Terk Eden')
axes[1, 0].set_title('Müşteri Süresi Dağılımı')
axes[1, 0].set_xlabel('Ay')
axes[1, 0].legend()

# 4. Aylık ücret dağılımı (terk durumuna göre)
df[df['Churn'] == 'No']['MonthlyCharges'].hist(ax=axes[1, 1], bins=30, alpha=0.6, 
                                                color='#2ecc71', label='Kalan')
df[df['Churn'] == 'Yes']['MonthlyCharges'].hist(ax=axes[1, 1], bins=30, alpha=0.6, 
                                                 color='#e74c3c', label='Terk Eden')
axes[1, 1].set_title('Aylık Ücret Dağılımı')
axes[1, 1].set_xlabel('$')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### 3.2 Ödeme Yöntemi ve İnternet Hizmeti Etkisi

İki önemli kategorik değişkenin terk oranına etkisini inceliyoruz:
- **Ödeme yöntemi:** Elektronik çek kullananlar daha mı çok terk ediyor?
- **İnternet hizmeti:** Fiber optik kullanıcılarında terk oranı neden yüksek olabilir?

In [ ]:
# Ödeme yöntemi ve internet hizmetine göre terk
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('PaymentMethod')['Churn_Flag'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Ödeme Yöntemine Göre Terk Oranı')
axes[0].set_ylabel('Terk Oranı')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=25)

df.groupby('InternetService')['Churn_Flag'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], color='#9b59b6', edgecolor='white')
axes[1].set_title('İnternet Hizmetine Göre Terk Oranı')
axes[1].set_ylabel('Terk Oranı')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 4. Özellik Mühendisliği ve Kodlama

### Neden Özellik Mühendisliği?

Ham veri doğrudan modele verilemez çünkü:
1. **Kategorik değişkenler** (Contract, PaymentMethod) metin formatında — model sadece sayıları anlar
2. Mevcut özelliklerden **yeni türetilmiş özellikler** oluşturmak modelin gücünü artırır

### Yapılan İşlemler

| İşlem | Açıklama |
|-------|----------|
| `LabelEncoder` | Kategorik sütunları sayıya çevirir (Aylık→0, 1 Yıl→1, 2 Yıl→2) |
| `AvgMonthlySpend` | TotalCharges / tenure → Aylık ortalama harcama |
| `IsNewCustomer` | tenure < 6 ay mı? → Yeni müşteri bayrağı |
| `HighSpender` | MonthlyCharges > medyan mı? → Yüksek harcama bayrağı |

> **Label Encoding** sıralı (ordinal) veriler için uygundur (sözleşme süresi gibi). Sırasız veriler için One-Hot Encoding tercih edilir, ama XGBoost ağaç tabanlı olduğu için Label Encoding genellikle yeterlidir.

In [ ]:
# Veriyi kopyala (orijinali bozmamak için)
df_model = df.copy()

# Kategorik değişkenleri LabelEncoder ile sayıya çevir
le_contract = LabelEncoder()
df_model['Contract_encoded'] = le_contract.fit_transform(df_model['Contract'])

le_payment = LabelEncoder()
df_model['PaymentMethod_encoded'] = le_payment.fit_transform(df_model['PaymentMethod'])

le_internet = LabelEncoder()
df_model['InternetService_encoded'] = le_internet.fit_transform(df_model['InternetService'])

le_tech = LabelEncoder()
df_model['TechSupport_encoded'] = le_tech.fit_transform(df_model['TechSupport'])

# Yeni türetilmiş özellikler
df_model['AvgMonthlySpend'] = df_model['TotalCharges'] / df_model['tenure'].clip(1)
df_model['IsNewCustomer'] = (df_model['tenure'] < 6).astype(int)
df_model['HighSpender'] = (df_model['MonthlyCharges'] > df_model['MonthlyCharges'].median()).astype(int)

# Model için kullanılacak özellikler
feature_cols = [
    'tenure', 'MonthlyCharges', 'TotalCharges',
    'Contract_encoded', 'PaymentMethod_encoded',
    'InternetService_encoded', 'TechSupport_encoded',
    'AvgMonthlySpend', 'IsNewCustomer', 'HighSpender'
]

X = df_model[feature_cols]
y = df_model['Churn_Flag']

print(f"Özellik sayısı: {len(feature_cols)}")
print(f"Özellikler: {feature_cols}")

### Eğitim ve Test Setlerine Ayırma

Veriyi %80 eğitim, %20 test olarak bölüyoruz. `stratify=y` ile her iki sette de terk oranının aynı kalmasını sağlıyoruz (dengesiz sınıf dağılımında bu çok önemli).

In [ ]:
# Eğitim-test ayırma
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Eğitim seti: {X_train.shape[0]} örnek")
print(f"Test seti:   {X_test.shape[0]} örnek")
print(f"\nEğitim seti terk oranı: {y_train.mean():.2%}")
print(f"Test seti terk oranı:   {y_test.mean():.2%}")

## 5. XGBoost Model Eğitimi

### XGBoost Nedir?

**XGBoost (eXtreme Gradient Boosting)**, zayıf öğrenicileri (küçük karar ağaçları) sırayla birleştirerek güçlü bir model oluşturan **ensemble** algoritmadır.

### Hiperparametre Açıklamaları

| Parametre | Değer | Açıklama |
|-----------|-------|----------|
| `n_estimators` | 200 | Kaç ağaç oluşturulacak (çok = güçlü ama yavaş) |
| `max_depth` | 4 | Her ağacın maksimum derinliği (yüksek = overfitting riski) |
| `learning_rate` | 0.1 | Her ağacın katkı oranı (düşük = daha dikkatli öğrenme) |
| `subsample` | 0.8 | Her ağaçta verinin %80'i kullanılır (overfitting önler) |
| `colsample_bytree` | 0.8 | Her ağaçta özelliklerin %80'i kullanılır |

> **Eğitim ve Test doğruluğu arasındaki fark** overfitting göstergesidir. Fark büyükse model eğitim verisini ezberlemiş demektir.

In [ ]:
# XGBoost modeli oluştur ve eğit
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

xgb_model.fit(X_train, y_train)

# Tahminler
y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:, 1]

print(f"Eğitim Doğruluğu: {xgb_model.score(X_train, y_train):.4f}")
print(f"Test Doğruluğu:   {accuracy_score(y_test, y_pred):.4f}")

## 6. Özellik Önemi (Feature Importance)

XGBoost her özelliğin model kararlarına ne kadar katkıda bulunduğunu hesaplar. Bu bilgi:
- Hangi faktörlerin terk davranışını en çok etkilediğini gösterir
- İş birimlerine aksiyon önerileri yapılmasını sağlar
- Gereksiz özelliklerin ayıklanmasına yardımcı olur

> En önemli özellik genellikle **sözleşme türü** ve **müşteri süresi (tenure)** olur — kısa süreli, aylık sözleşmeli müşteriler en yüksek terk riskini taşır.

In [ ]:
# Özellik önem grafiği (yatay çubuk)
imp_df = pd.DataFrame({
    'Özellik': feature_cols,
    'Önem': xgb_model.feature_importances_
}).sort_values('Önem', ascending=True)

plt.figure(figsize=(10, 7))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(imp_df)))
plt.barh(imp_df['Özellik'], imp_df['Önem'], color=colors, edgecolor='white')
plt.xlabel('Önem Değeri')
plt.title('XGBoost — Özellik Önem Sıralaması')
plt.tight_layout()
plt.show()

print("\nEn önemli 5 özellik:")
for _, row in imp_df.tail(5).iloc[::-1].iterrows():
    print(f"  {row['Özellik']:30s} → {row['Önem']:.4f}")

## 7. Model Değerlendirme

### 7.1 Karışıklık Matrisi

Modelin tahminlerini 4 kategoride inceliyoruz:

|  | Tahmin: Kalan | Tahmin: Terk |
|--|--------------|-------------|
| **Gerçek: Kalan** | TN (Doğru) | FP (Yanlış alarm) |
| **Gerçek: Terk** | FN (Kaçırılan müşteri) | TP (Doğru terk tahmini) |

İş açısından:
- **FP (Yanlış alarm):** Terk etmeyecek müşteriye gereksiz indirim sunma → düşük maliyet
- **FN (Kaçırma):** Terk edecek müşteriyi fark etmeme → **yüksek maliyet** (müşteri kaybı)

In [ ]:
# Karışıklık matrisi ısı haritası
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Kalan', 'Terk Eden'],
            yticklabels=['Kalan', 'Terk Eden'])
plt.title('XGBoost — Karışıklık Matrisi')
plt.xlabel('Tahmin')
plt.ylabel('Gerçek')
plt.tight_layout()
plt.show()

# Sınıflandırma raporu
print("\nSınıflandırma Raporu:")
print(classification_report(y_test, y_pred, target_names=['Kalan', 'Terk Eden']))

### 7.2 ROC Eğrisi ve AUC

ROC eğrisi, farklı eşik değerlerinde modelin **Doğru Pozitif Oranı (TPR)** ile **Yanlış Pozitif Oranı (FPR)** arasındaki dengeyi gösterir.

- **AUC > 0.80:** İyi bir terk tahmin modeli
- **AUC > 0.85:** Üretim ortamına alınabilir düzeyde
- **AUC > 0.90:** Mükemmel

In [ ]:
# ROC eğrisi
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#2196F3', linewidth=2.5, 
         label=f'XGBoost (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.7)
plt.xlabel('Yanlış Pozitif Oranı')
plt.ylabel('Doğru Pozitif Oranı')
plt.title('ROC Eğrisi — Müşteri Terk Tahmini')
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. İş Analitiği ve Öneriler

### Temel Bulgular

#### 1. Sözleşme Türü En Belirleyici Faktör
- **Aylık sözleşme** yapan müşteriler en yüksek terk oranına sahiptir
- **2 yıllık sözleşme** yapan müşterilerde terk oranı çok düşüktür
- **Öneri:** Aylık müşterilere yıllık sözleşmeye geçiş için indirim kampanyaları sunulmalı

#### 2. Yeni Müşteriler Risk Altında
- İlk 12 ayda terk oranı belirgin şekilde yüksektir
- **Öneri:** Yeni müşterilere özel ilk yıl sadakat programları tasarlanmalı

#### 3. Ödeme Yöntemi Etkisi
- **Elektronik çek** ile ödeme yapan müşterilerde terk oranı daha yüksektir
- **Otomatik ödeme** kullanan müşterilerde terk oranı düşüktür
- **Öneri:** Otomatik ödemeye geçmeleri için teşvikler sunulmalı

#### 4. Aylık Ücret ve Teknik Destek
- Yüksek aylık ücret ödeyen müşterilerde terk riski artmaktadır
- Teknik destek alan müşterilerde terk oranı daha düşüktür
- **Öneri:** Yüksek ücret ödeyenlere ekstra değer sunulmalı

### Aksiyon Planı

| Öncelik | Aksiyon | Hedef Kitle | Beklenen Etki |
|---------|---------|-------------|---------------|
| Yüksek | Sözleşme geçiş kampanyası | Aylık müşteriler | Terk oranında %15-20 düşüş |
| Yüksek | İlk yıl sadakat programı | Yeni müşteriler (0-12 ay) | Erken terk oranında %10 düşüş |
| Orta | Otomatik ödeme teşviki | Elektronik çek kullananlar | Terk oranında %5-8 düşüş |
| Orta | Değer paketi oluşturma | Yüksek ücret ödeyenler | Müşteri memnuniyetinde artış |
| Düşük | Teknik destek iyileştirme | Tüm müşteriler | Uzun vadeli terk azalması |

### Alıştırmalar
1. `learning_rate` değerini 0.01 ve 0.3 yaparak sonuçları karşılaştırın
2. `max_depth` parametresini 2, 6, 10 yaparak overfitting etkisini gözlemleyin
3. **Bonus:** Random Forest ve LightGBM modelleri ekleyip XGBoost ile karşılaştırın
4. Eşik değerini 0.5 yerine 0.3 yaparak Recall'u artırmaya çalışın (daha az müşteri kaçırma)

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

© 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>